ValueError: could not convert string to float: 'The reactor was pressurized with nitrogen to 2.0 MPa at room temperature before heating, ensuring an inert atmosphere during the reaction'

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset

data = pd.read_csv("data.csv")

# Select relevant features and target variable
columns_of_interest = [
    'pressure(process)', 'catalyst used and its specification', 
    'Metal Loading (%)', 'Support Material', 
    'Surface Area (m²/g)', 'Reaction Time (h)', 
    'Lignin Loading (wt%)', 'Conversion rate1'
]



In [19]:
model_data = data[columns_of_interest]

# Fill missing values
model_data['pressure(process)'].fillna('Unknown', inplace=True)
model_data['catalyst used and its specification'].fillna('Unknown', inplace=True)



C:\Users\Purushoth\AppData\Local\Temp\ipykernel_952\53220564.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  model_data['pressure(process)'].fillna('Unknown', inplace=True)
C:\Users\Purushoth\AppData\Local\Temp\ipykernel_952\53220564.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_data['pressure(process)'].fillna('Unknown', inpla

In [20]:
# Convert `pressure(process)` to numeric
def simplify_pressure(pressure):
    try:
        if 'MPa' in str(pressure):
            return float(str(pressure).replace(' MPa', ''))
        elif 'atmospheric' in str(pressure).lower():
            return 0.1  # Approximate atmospheric pressure in MPa
        elif pressure == 'Unknown' or 'not specified' in str(pressure).lower():
            return np.nan
        else:
            return np.nan  # Handle unstructured text
    except:
        return np.nan

model_data['pressure(process)'] = model_data['pressure(process)'].apply(simplify_pressure)

# Encode categorical variables


C:\Users\Purushoth\AppData\Local\Temp\ipykernel_952\3865063297.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_data['pressure(process)'] = model_data['pressure(process)'].apply(simplify_pressure)


In [22]:
categorical_features = ['catalyst used and its specification', 'Support Material']
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cats = encoder.fit_transform(model_data[categorical_features])
encoded_cat_columns = encoder.get_feature_names_out(categorical_features)
encoded_cat_df = pd.DataFrame(encoded_cats, columns=encoded_cat_columns, index=model_data.index)

# Merge encoded data and drop original categorical columns


In [23]:
model_data = model_data.drop(columns=categorical_features).join(encoded_cat_df)

# Drop rows with missing values
model_data.dropna(inplace=True)

# Split data into features and target
X = model_data.drop(columns=['Conversion rate1'])
y = model_data['Conversion rate1']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Regressor
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate the model
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("R-squared:", r2)

Mean Squared Error: 701.190400000001
R-squared: nan


c:\Users\Purushoth\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:1211: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [25]:
# Define input as variables
input_data = {
    'pressure(process)': [5.0],  # Example pressure in MPa
    'Metal Loading (%)': [2.5],  # Example Metal Loading
    'Surface Area (m²/g)': [150],  # Example Surface Area
    'Reaction Time (h)': [3],  # Example Reaction Time
    'Lignin Loading (wt%)': [10],  # Example Lignin Loading
}

# Specify catalyst and support material
input_catalyst = 'Example Catalyst'
input_support = 'Example Support'

# Encode categorical features
for col in encoded_cat_columns:
    input_data[col] = [1 if col.endswith(input_catalyst) or col.endswith(input_support) else 0]

# Create a DataFrame for prediction
input_df = pd.DataFrame(input_data)

# Predict and print the result
prediction = model.predict(input_df)
print(f"Predicted Conversion Rate: {prediction[0]}")

Predicted Conversion Rate: 448.43
